# Choose data for slideflow

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from glob import glob

dirpath = Path.cwd()
print(dirpath)

## Load data

In [ ]:
datapath = dirpath/'../data'
metapath = datapath/'meta'
df = pd.read_csv(metapath/'meta_merged.csv')
print(df.shape)
display(df[:2])

In [ ]:
c = 'tumor_site_from_data_src'
print(df[c].nunique())
tt = df.groupby([c]).agg({'model': 'nunique',
                          'patient_id': 'nunique',
                          'specimen_id': 'nunique',
                          'image_id': 'nunique'}).reset_index()
display(tt)

In [ ]:
c = 'tumor_type_from_data_src'
print(df[c].nunique())
tt = df.groupby(['tumor_type_from_data_src']).agg({'model': 'nunique',
                                                   'patient_id': 'nunique', 
                                                   'specimen_id': 'nunique',
                                                   'image_id': 'nunique'}).reset_index()
display(tt)

## Choose the tissue/cancer types

In [ ]:
# types = ['Melanoma', 'Salivary gland cancer']
types = ['Melanoma', 'Salivary gland cancer']
aa = df[ df['tumor_type_from_data_src'].isin(types) ]
display(aa)

In [ ]:
# Create balanced dataset that contains n number of samples from each type
n = 5
bb1 = aa[ aa['tumor_type_from_data_src'] == types[0] ][:n]
bb2 = aa[ aa['tumor_type_from_data_src'] == types[1] ][:n]
df_meta = pd.concat([bb1, bb2]).reset_index(drop=True)
df_meta

In [ ]:
sorted(df_meta['image_id'])

# Copy image metadata to pdx-proj for SlideFlow

In [ ]:
import shutil

In [ ]:
# src_img_path = datapath/'svs_images'
src_img_path = datapath/'doe-globus-pdx-data'
dst_img_path = Path('/Users/apartin/work/jdacs/slideflow-proj/sf_pdx_proj/slides')  # Mac

In [ ]:
for fname in df_meta['image_id'].values:
    _ = shutil.copyfile(str(src_img_path/f'{fname}.svs'), str(dst_img_path/f'{fname}.svs'))

# Create files for slideflow

In [ ]:
df_meta.sort_values('image_id')

### annotations.csv

In [ ]:
annt = df_meta.copy()
# annt.insert(loc=0, column='submitter_id', value=df_meta['patient_id'], allow_duplicates=True)
annt.insert(loc=0, column='submitter_id', value=range(len(df_meta)), allow_duplicates=True)
annt.insert(loc=1, column='slide', value=df_meta['image_id'], allow_duplicates=True)

In [ ]:
annt

In [ ]:
annt.to_csv('/Users/apartin/work/jdacs/slideflow-proj/pdx-proj/project/annotations.csv', index=False)  # Mac